# Plot Daily Profile

Plot one home's average hourly consumption for January-August 2013.

Open this notebook in `notebooks/M5/`. Run the cells in order. Saving again replaces this notebook's output files.


## Shared folder setup

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

## Read the M2 hourly file

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
hourly_path = PROJECT_ROOT / "data/pilot/hourly_sample.csv"
hourly = pd.read_csv(hourly_path)
hourly.head()

## Parse the needed columns

In [ ]:
hourly["hour_start"] = pd.to_datetime(hourly["hour_start"], utc=True)
hourly["kWh"] = pd.to_numeric(hourly["kWh"])
hourly.dtypes

## Select the same example home and TRAIN dates

In [ ]:
sample_home = "MAC000414"
start = pd.Timestamp("2013-01-01", tz="UTC")
end = pd.Timestamp("2013-09-01", tz="UTC")
selected = hourly[(hourly["LCLid"] == sample_home) & (hourly["hour_start"] >= start) & (hourly["hour_start"] < end)]
selected.head()

## Make the full expected hourly grid

In [ ]:
hours = pd.date_range(start, end, freq="h", inclusive="left")
values = selected.set_index("hour_start")["kWh"].reindex(hours)
len(values), values.isna().sum()

## Separate clock hour from date

In [ ]:
profile_input = values.rename("kWh").to_frame()
profile_input["hour_of_day"] = profile_input.index.hour
profile_input.head()

## Calculate the mean and available observations

In [ ]:
profile = profile_input.groupby("hour_of_day")["kWh"].agg(mean_hourly_kwh="mean",
    available_observations="count")
profile = profile.reindex(range(24)).reset_index()
profile.insert(0, "LCLid", sample_home)
profile

## Show coverage alongside the mean

In [ ]:
days = (end - start).days
coverage = profile[["LCLid", "hour_of_day", "available_observations"]].copy()
coverage["expected_observations"] = days
coverage["missing_observations"] = days - coverage["available_observations"]
coverage

## Read the range of available days

In [ ]:
min_obs = profile["available_observations"].min()
max_obs = profile["available_observations"].max()
min_obs, max_obs, days

## Chart 3: average daily profile

In [ ]:
profile_fig, profile_ax = plt.subplots(figsize=(9, 4.5))
profile_ax.plot(profile["hour_of_day"], profile["mean_hourly_kwh"], marker="o", color="#0056b3")
profile_ax.set(title="Example home: daily profile, TRAIN Jan-Aug 2013",
    xlabel="Hour of day (UTC)", ylabel="Mean hourly energy (kWh)")
profile_ax.set_xticks(range(24))
profile_ax.set_ylim(bottom=0)
profile_fig.tight_layout()
plt.show()

## Find the highest and lowest mean hours

In [ ]:
peak = profile.loc[profile["mean_hourly_kwh"].idxmax()]
low = profile.loc[profile["mean_hourly_kwh"].idxmin()]
profile.loc[[low.name, peak.name], ["hour_of_day", "mean_hourly_kwh", "available_observations"]]

## Choose the output folder

In [ ]:
output_dir = PROJECT_ROOT / 'data/processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

## Save the four-column profile format

In [ ]:
profile_path = output_dir / "daily_profile_sample.csv"
profile.to_csv(profile_path, index=False)
profile_path

## Save the profile chart

In [ ]:
chart_path = PROJECT_ROOT / "figures" / "daily_profile_sample.png"
profile_fig.savefig(chart_path, dpi=150)
chart_path

## Save the hourly coverage table

In [ ]:
coverage_path = output_dir / "daily_profile_coverage_sample.csv"
coverage.to_csv(coverage_path, index=False)
coverage_path

## Prepare a scope and coverage caption

In [ ]:
caption = (
    f"Home {sample_home}, Jan-Aug 2013. Mean kWh for each hour of the day. "
    f"Each hour uses {min_obs}-{max_obs} available days out of {days}. Missing hours are excluded."
)
caption

## Save the profile notes

In [ ]:
notes_path = PROJECT_ROOT / "figures" / "daily_profile_notes_sample.md"
notes = "# M5 daily profile notes\n\n" + caption + "\n\nThe profile shows one home during the training period.\n"
notes_path.write_text(notes, encoding="utf-8")
notes_path

## Open the saved profile

In [ ]:
pd.read_csv(profile_path)